# Few-Shot Cross-Domain 3D Abdominal Multi-Organ Segmentation
## 🚀 Google Colab & Kaggle GPU Benchmark Runner (Hybrid Swin-UNet vs. 3D U-Net)

This notebook runs the complete few-shot cross-domain adaptation benchmark on cloud GPUs (T4, P100, V100, A100):
- **Source Domain (Few-Shot Support):** AMOS 2022 CT ($k=1, 5$ shots streamed on-demand from Hugging Face)
- **Target Domain (Zero-Shot Generalization):** BTCV CT (30 cases streamed on-demand from Hugging Face)
- **13 Overlapping Organs:** Spleen, R. Kidney, L. Kidney, Gallbladder, Esophagus, Liver, Stomach, Aorta, IVC, Portal/Splenic Vein, Pancreas, R. Adrenal, L. Adrenal
- **Metrics:** Dice Similarity Coefficient (DSC %), 95% Hausdorff Distance (HD95 mm), Average Surface Distance (ASD mm)

### Step 1: Environment & GPU Verification

In [ ]:
# Install lightweight dependencies (NIfTI reader & HuggingFace Hub)
!pip install -q nibabel huggingface_hub scipy tqdm pandas matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Step 2: Clone or Load Workspace Code

In [ ]:
import sys, os
# If running in Colab and you uploaded the repo folder or cloned via git:
if os.path.exists("Hybrid_Swin_UNet"):
    %cd Hybrid_Swin_UNet
sys.path.append(os.getcwd())
print("Current Directory:", os.getcwd())

### Step 3: Run Few-Shot Adaptation Benchmark
Run both the **Proposed Hybrid Swin-UNet** and the **3D U-Net Baseline** under identical few-shot protocols ($k=1, 5$ shots, 50 epochs).

In [ ]:
# 1. Run 5-Shot Adaptation for Hybrid Swin-UNet
!python train.py --model hybrid_swin --shots 5 --epochs 50 --eval_cases 10 --seed 42

# 2. Run 5-Shot Adaptation for 3D U-Net Baseline
!python train.py --model unet3d --shots 5 --epochs 50 --eval_cases 10 --seed 42

### Step 4: Run Multi-Regime Benchmark ($k=1, 3, 5, 10$ Shots & Multiple Seeds)

In [ ]:
from scripts.train_fewshot import run_experiment

regimes = [1, 3, 5, 10]
models = ["hybrid_swin", "unet3d"]

for m in models:
    for k in regimes:
        print(f"\n>>> Running {m.upper()} on {k}-Shot Adaptation...")
        run_experiment(model_type=m, k_shots=k, num_epochs=30, seed=42, eval_cases=5)